# Pattern 7: Tool-Side External Authorization (OPA ReBAC)

On the MCP side, this is identical to pattern 5: JWT passthrough. The difference is on the service side: before executing an action, the service calls OPA with a relationship-based access control (ReBAC) query.

OPA evaluates rules like: admin override, self-access, manager-of-target, department peer read.

**What the service sees:** Proven identity + per-resource authorization decisions.

**Weakness:** The agent obtained the user's token via direct grant (password flow). The user never explicitly consented to the agent acting on their behalf.

In [1]:
from framework.runner import PatternRunner
runner = PatternRunner("p07_external_authz_tool")

## The auth code

This is the lesson. Read both files to understand what happens at each boundary.

In [2]:
runner.show_auth_code()

╭───────────────────── p07_external_authz_tool/mcp_auth.py ─────────────────────╮
│    1 """Pattern 7: Tool-Side External Authorization (OPA ReBAC).              │
│    2                                                                          │
│    3 On the MCP side, this is identical to pattern 5: forward the user's JWT  │
│    4 (received from the agent caller) as a Bearer token. The fine-grained OPA │
│    5 check happens on the service side.                                       │
│    6 """                                                                      │
│    7                                                                          │
│    8 from framework.mcp.auth import AuthHandler                               │
│    9                                                                          │
│   10                                                                          │
│   11 class JWTPassthroughHandler(AuthHandler):                                │
│   12     async def prepare_request(self, user_context, headers):              │
│   13         jwt = user_context.get("jwt")                                    │
│   14         if jwt:                                                          │
│   15             headers["Authorization"] = f"Bearer {jwt}"                   │
│   16         return headers                                                   │
│   17                                                                          │
│   18                                                                          │
│   19 auth_handler = JWTPassthroughHandler()                                   │
│   20                                                                          │
╰───────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────── p07_external_authz_tool/service_auth.py ────────────────────────────────────╮
│    1 """Pattern 7: Tool-Side External Authorization (service side).                                             │
│    2                                                                                                            │
│    3 Same JWKS-based JWT validation as pattern 5. The addition: the OPA URL is                                  │
│    4 stored in the identity claims so the service route (approve_expense) can                                   │
│    5 call OPA for per-resource authorization before executing the action.                                       │
│    6                                                                                                            │
│    7 OPA evaluates relationship-based rules (tool_side.rego): admin override,                                   │
│    8 self-access, manager-of-target, department peer read.                                                      │
│    9 """                                                                                                        │
│   10                                                                                                            │
│   11 import jwt as pyjwt                                                                                        │
│   12 from fastapi import Request                                                                                │
│   13 from jwt import PyJWKClient                                                                                │
│   14                                                                                                            │
│   15 from framework.config import (                                                                             │
│   16     EXPECTED_ISSUER,                                                                                       │
│   17     EXPENSE_SERVICE_CLIENT_ID,                                                                             │
│   18     DOCUMENT_SERVICE_CLIENT_ID,                                                                            │
│   19     JWKS_URL,                                                                                              │
│   20     OPA_URL,                                                                                               │
│   21 )                                                                                                          │
│   22 from framework.services.identity import Identity                                                           │
│   23                                                                                                            │
│   24 _expense_jwk_client: PyJWKClient | None = None                                                             │
│   25 _document_jwk_client: PyJWKClient | None = None                                                            │
│   26                                                                                                            │
│   27                                                                                                            │
│   28 async def get_expense_identity(request: Request) -> Identity:                                              │
│   29     """Validate JWT + store OPA URL for tool-side authz."""                                                │
│   30     identity = await _validate_jwt(request, EXPENSE_SERVICE_CLIENT_ID, "_expense")                         │
│   31     if identity.claims is not None:                                                                        │
│   32         identity.claims["_opa_url"] = OPA_URL                                                              │
│   33     return identity                                                                                        │
│   34                                                  

The MCP side is identical to pattern 5: forward the JWT.
The service validates the JWT AND calls OPA for per-resource authorization.
OPA checks relationship-based rules before allowing the action.

### What's in the OPA policy?

The tool-side policy (`infra/opa/tool_side.rego`) uses relationship-based access control (ReBAC). Unlike the role-based agent-side policy in pattern 4, this one checks *who owns the resource* being accessed:

```rego
# Default deny.
default allow := false

# Admins can do anything.
allow if { input.user_id in data.admins }

# Users can always read their own resources.
allow if {
    input.action == "read"
    input.user_id == input.target_user_id
}

# Managers can read AND approve resources owned by their direct reports.
allow if {
    input.action in {"read", "approve"}
    reports := object.get(data.relationships.manages, input.user_id, [])
    input.target_user_id in reports
}

# Members of the same department can read each other's resources.
allow if {
    input.action == "read"
    some department
    input.user_id in data.relationships.members[department]
    input.target_user_id in data.relationships.members[department]
}
```

The input comes from the service: `{user_id, target_user_id, action, resource_type}`. The relationships (`manages`, `members`, `admins`) are loaded from `infra/opa/data.json`. The full policy is in `infra/opa/tool_side.rego`.

### Contrast with pattern 4

Pattern 4's agent-side OPA asks "can this *role* use this *tool*?" -- a coarse check.
Pattern 7's tool-side OPA asks "can this *user* act on this *specific resource*?" -- a fine-grained check.
Both use OPA, but the inputs and granularity are fundamentally different.

In [3]:
await runner.start()

[04/14/26 06:38:43] INFO     StreamableHTTP session manager started                  ]8;id=520773;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=440838;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     StreamableHTTP session manager started                  ]8;id=319794;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=250099;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#128\128]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:51271/mcp "HTTP/1.1 200 OK"        ]8;id=405994;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=247739;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     Negotiated protocol version: 2025-11-25                         ]8;id=569385;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py\streamable_http.py]8;;\:]8;id=513410;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/client/streamable_http.py#193\193]8;;\

                    INFO     Terminating session: None                                       ]8;id=389607;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=597053;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

                    INFO     HTTP Request: POST http://127.0.0.1:51271/mcp "HTTP/1.1 200 OK"        ]8;id=694802;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=621059;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:51271/mcp "HTTP/1.1 200 OK"        ]8;id=393099;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=42400;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

                    INFO     HTTP Request: POST http://127.0.0.1:51271/mcp "HTTP/1.1 200 OK"        ]8;id=65639;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=392249;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\

[04/14/26 06:39:35] INFO     StreamableHTTP session manager shutting down            ]8;id=205424;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=830041;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

                    INFO     StreamableHTTP session manager shutting down            ]8;id=799564;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py\streamable_http_manager.py]8;;\:]8;id=792766;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http_manager.py#132\132]8;;\

## Run scenarios

Let's see this pattern in action with different users.

In [4]:
await runner.run_as("alice", "What are my expenses?")

[04/14/26 06:38:44] INFO     HTTP Request: POST                                                     ]8;id=730196;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=959306;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭───────────────────────────────╮
│ [alice] What are my expenses? │
╰───────────────────────────────╯

                    INFO     Terminating session: None                                       ]8;id=248416;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=942244;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

                    INFO     Processing request of type ListToolsRequest                              ]8;id=286975;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=352763;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

                    INFO     Terminating session: None                                       ]8;id=89058;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py\streamable_http.py]8;;\:]8;id=680604;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/streamable_http.py#785\785]8;;\

...

[04/14/26 06:38:49] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=939335;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=616139;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:38:53] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=855853;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=125267;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool         ┃ args ┃ status ┃ result                                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ get_expenses │ {}   │      - │ {'type': 'text', 'text': '{"identity_method": "jwt", "identity_detail":    │
│     │              │      │        │ "validated JWT issued by http://localhost:8080/realms/agentauth;           │
│     │              │      │        │ aud=[\'document-service-client\', \'expense-service-client\'];             │
│     │              │      │        │ azp=agent-client", "count": 4, "expenses": [{"id": 1, "user_id": "alice",  │
│     │              │      │        │ "department": "engineering                                                 │
└─────┴──────────────┴──────┴────────┴────────────────────────────────────────────────────────────────────────────┘

╭────────────────────────────────── answer ───────────────────────────────────╮
│ Here are your expenses (alice, engineering):                                │
│                                                                             │
│ - ID 1 | $42.50 | software | JetBrains AI assistant subscription | approved │
│ - ID 2 | $156.00 | travel | Train ticket to client offsite | approved       │
│ - ID 3 | $89.00 | books | Designing Data-Intensive Applications | approved  │
│ - ID 4 | $1,450.00 | hardware | External 4K monitor | pending               │
│                                                                             │
│ Totals:                                                                     │
│ - Approved: $287.50 (3 items)                                               │
│ - Pending: $1,450.00 (1 item)                                               │
│ - Grand total: $1,737.50                                                    │
│                                                                             │
│ Want me to filter by department, export, or approve any pending expense?    │
╰─────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Here are your expenses (alice, engineering):\n\n- ID 1 | $42.50 | software | JetBrains AI assistant subscription | approved\n- ID 2 | $156.00 | travel | Train ticket to client offsite | approved\n- ID 3 | $89.00 | books | Designing Data-Intensive Applications | approved\n- ID 4 | $1,450.00 | hardware | External 4K monitor | pending\n\nTotals:\n- Approved: $287.50 (3 items)\n- Pending: $1,450.00 (1 item)\n- Grand total: $1,737.50\n\nWant me to filter by department, export, or approve any pending expense?', tool_calls=[ToolCallTrace(name='get_expenses', args={}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "jwt", "identity_detail": "validated JWT issued by http://localhost:8080/realms/agentauth; aud=[\\\'document-service-client\\\', \\\'expense-service-client\\\']; azp=agent-client", "count": 4, "expenses": [{"id": 1, "user_id": "alice", "department": "engineering', error=None)])

In [5]:
# Expected: OPA denies this -- self-approval is not allowed
await runner.run_as("alice", "Approve expense 4")

                    INFO     HTTP Request: POST                                                     ]8;id=226337;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=130671;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭───────────────────────────╮
│ [alice] Approve expense 4 │
╰───────────────────────────╯

[04/14/26 06:38:54] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=106218;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=392131;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:38:56] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=576081;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=921098;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=302271;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=444525;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

...

[04/14/26 06:38:59] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=898234;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=844415;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:39:07] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=28366;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=216487;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool            ┃ args              ┃ status ┃ result                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ approve_expense │ {'expense_id': 4} │      - │ {'type': 'text', 'text': '{"detail": "role=employee cannot │
│     │                 │                   │        │ approve expenses", "_status": 403}'}                       │
└─────┴─────────────────┴───────────────────┴────────┴────────────────────────────────────────────────────────────┘

╭──────────────────────────────────────────────────── answer ─────────────────────────────────────────────────────╮
│ Approval failed: role=employee cannot approve expenses. You need a manager or admin account to approve expense  │
│ 4.                                                                                                              │
│                                                                                                                 │
│ If you think you should have approval rights, please use an approver’s account or contact your administrator to │
│ approve this. I can help by:                                                                                    │
│ - identifying your designated approver (if you share your department/role)                                      │
│ - listing expenses awaiting approval for your team (if you want)                                                │
│                                                                                                                 │
│ What would you like me to do next?                                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

AgentResult(content='Approval failed: role=employee cannot approve expenses. You need a manager or admin account to approve expense 4.\n\nIf you think you should have approval rights, please use an approver’s account or contact your administrator to approve this. I can help by: \n- identifying your designated approver (if you share your department/role)\n- listing expenses awaiting approval for your team (if you want)\n\nWhat would you like me to do next?', tool_calls=[ToolCallTrace(name='approve_expense', args={'expense_id': 4}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"detail": "role=employee cannot approve expenses", "_status": 403}\'}', error=None)])

In [6]:
# Expected: OPA allows this -- bob manages alice
await runner.run_as("bob", "Approve expense 4")

                    INFO     HTTP Request: POST                                                     ]8;id=389248;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=440751;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭─────────────────────────╮
│ [bob] Approve expense 4 │
╰─────────────────────────╯

[04/14/26 06:39:09] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=229289;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=779232;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=539276;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=972649;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=627473;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=224183;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8181/v1/data/agentauth/tool_side/decision "HTTP/1.1                  
                             200 OK"                                                                               

...

[04/14/26 06:39:10] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=350049;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=623641;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:39:27] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=670059;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=87388;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool            ┃ args              ┃ status ┃ result                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ approve_expense │ {'expense_id': 4} │      - │ {'type': 'text', 'text': '{"identity_method": "jwt",       │
│     │                 │                   │        │ "approved_by": "bob", "tool_side_authz_enabled": true,     │
│     │                 │                   │        │ "expense": {"id": 4, "user_id": "alice", "department":     │
│     │                 │                   │        │ "engineering", "amount": 1450.0, "category": "hardware",   │
│     │                 │                   │        │ "description": "External 4K monitor", "status":            │
│     │                 │                   │        │ "approved"}, "_status": 200}'}                             │
└─────┴─────────────────┴───────────────────┴────────┴────────────────────────────────────────────────────────────┘

╭───────────────── answer ─────────────────╮
│ Expense 4 approved.                      │
│                                          │
│ Details:                                 │
│ - User: alice (engineering)              │
│ - Amount: 1,450.00                       │
│ - Category: hardware                     │
│ - Description: External 4K monitor       │
│ - Approved by: bob                       │
│ - Status: approved                       │
│                                          │
│ Need to review or act on other expenses? │
╰──────────────────────────────────────────╯

AgentResult(content='Expense 4 approved.\n\nDetails:\n- User: alice (engineering)\n- Amount: 1,450.00\n- Category: hardware\n- Description: External 4K monitor\n- Approved by: bob\n- Status: approved\n\nNeed to review or act on other expenses?', tool_calls=[ToolCallTrace(name='approve_expense', args={'expense_id': 4}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "jwt", "approved_by": "bob", "tool_side_authz_enabled": true, "expense": {"id": 4, "user_id": "alice", "department": "engineering", "amount": 1450.0, "category": "hardware", "description": "External 4K monitor", "status": "approved"}, "_status": 200}\'}', error=None)])

In [7]:
# Expected: OPA allows this -- dave is admin
await runner.run_as("dave", "Approve expense 5")

                    INFO     HTTP Request: POST                                                     ]8;id=40470;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=714729;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8080/realms/agentauth/protocol/openid-connect/token                  
                             "HTTP/1.1 200 OK"                                                                     

╭──────────────────────────╮
│ [dave] Approve expense 5 │
╰──────────────────────────╯

[04/14/26 06:39:29] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=291973;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=894177;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                    INFO     Processing request of type CallToolRequest                               ]8;id=331052;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py\server.py]8;;\:]8;id=513508;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/mcp/server/lowlevel/server.py#727\727]8;;\

                    INFO     HTTP Request: POST                                                     ]8;id=940016;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=235965;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             http://localhost:8181/v1/data/agentauth/tool_side/decision "HTTP/1.1                  
                             200 OK"                                                                               

...

[04/14/26 06:39:30] INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=431302;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=627924;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       

[04/14/26 06:39:34] INFO     HTTP Request: POST https://api.openai.com/v1/responses "HTTP/1.1 200   ]8;id=781924;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=553717;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             OK"                                                                                   

                                                    tool calls                                                     
┏━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ tool            ┃ args              ┃ status ┃ result                                                     ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ approve_expense │ {'expense_id': 5} │      - │ {'type': 'text', 'text': '{"identity_method": "jwt",       │
│     │                 │                   │        │ "approved_by": "dave", "tool_side_authz_enabled": true,    │
│     │                 │                   │        │ "expense": {"id": 5, "user_id": "bob", "department":       │
│     │                 │                   │        │ "engineering", "amount": 320.0, "category": "training",    │
│     │                 │                   │        │ "description": "OAuth 2.0 deep-dive workshop", "status":   │
│     │                 │                   │        │ "approved"}, "_status": 2                                  │
└─────┴─────────────────┴───────────────────┴────────┴────────────────────────────────────────────────────────────┘

╭───────────────────────────── answer ──────────────────────────────╮
│ Expense 5 is already approved.                                    │
│                                                                   │
│ Details:                                                          │
│ - Approved by: dave                                               │
│ - User: bob (engineering)                                         │
│ - Amount: 320.00                                                  │
│ - Category: training                                              │
│ - Description: OAuth 2.0 deep-dive workshop                       │
│                                                                   │
│ Would you like to view other expenses or approve a different one? │
╰───────────────────────────────────────────────────────────────────╯

AgentResult(content='Expense 5 is already approved.\n\nDetails:\n- Approved by: dave\n- User: bob (engineering)\n- Amount: 320.00\n- Category: training\n- Description: OAuth 2.0 deep-dive workshop\n\nWould you like to view other expenses or approve a different one?', tool_calls=[ToolCallTrace(name='approve_expense', args={'expense_id': 5}, status=None, result_summary='{\'type\': \'text\', \'text\': \'{"identity_method": "jwt", "approved_by": "dave", "tool_side_authz_enabled": true, "expense": {"id": 5, "user_id": "bob", "department": "engineering", "amount": 320.0, "category": "training", "description": "OAuth 2.0 deep-dive workshop", "status": "approved"}, "_status": 2', error=None)])

## What did the service see?

The punchline: what identity did the backend service actually extract?

In [8]:
await runner.show_service_identity()

                    INFO     HTTP Request: GET http://127.0.0.1:51269/debug/last-request "HTTP/1.1  ]8;id=666323;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=807609;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭────────────────────────────────────── expense-service /debug/last-request ──────────────────────────────────────╮
│ method:  jwt                                                                                                    │
│ user_id: dave                                                                                                   │
│ detail:  validated JWT issued by http://localhost:8080/realms/agentauth; aud=['document-service-client',        │
│ 'expense-service-client']; azp=agent-client                                                                     │
│                                                                                                                 │
│ claims:  sub=bc99f071..., role=admin, department=platform, aud=['document-service-client',                      │
│ 'expense-service-client'], azp=agent-client                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                    INFO     HTTP Request: GET http://127.0.0.1:51270/debug/last-request "HTTP/1.1  ]8;id=13352;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=302528;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1740\1740]8;;\
                             200 OK"                                                                               

╭─ document-service /debug/last-request ─╮
│ method:  never                         │
│ user_id: <none>                        │
│ detail:  no requests yet               │
│                                        │
╰────────────────────────────────────────╯

### What changed from patterns 5-6

In patterns 5-6, any manager could approve any expense in their department -- the service checked the role but not the relationship. Now OPA enforces per-resource rules: alice cannot approve her own expense (self-approval denied), bob can approve alice's expense (manager-of-target), dave can approve anything (admin override).

Alice's attempt to approve her own expense was denied by OPA. Bob can approve Alice's expense because the relationship data (`data.relationships.manages`) shows bob manages alice. Dave can approve anything as admin.

This is fine-grained, relationship-based authorization enforced at the service level. The service checks "is this manager allowed to approve THIS specific expense?", not just "is this user a manager?".

### The weakness

One weakness remains: the agent obtained the token via direct grant (it knows the user's password). The user never explicitly consented to the agent acting on their behalf.

In [9]:
await runner.stop()

Pattern p07_external_authz_tool stopped.

                    INFO     HTTP Request: POST https://api.openai.com/v1/traces/ingest "HTTP/1.1   ]8;id=867619;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=626791;file:///Users/carloperottino/_develop/exploring-agent-auth/.venv/lib/python3.12/site-packages/httpx/_client.py#1025\1025]8;;\
                             204 No Content"                                                                       